# 앙상블

앙상블은 여러 모델의 예측을 결합해 더 안정적이고 강한 예측을 만들려는 방법이다.

1. 모델 하나가 항상 최고인 것은 아니다.
2. 서로 다른 성향의 모델을 함께 쓰면 한 모델의 약점을 다른 모델이 보완할 수 있다.
3. 앙상블의 목적은 **무조건 복잡하게 만들기**가 아니라 **예측을 더 안정적으로 만들기**이다.

대표적인 앙상블 계열은 다음과 같다.

1. Voting
   - 여러 모델의 예측을 모아 최종 결과를 결정한다.
   - 분류에서는 hard voting, soft voting을 주로 사용한다.
2. Bagging
   - 같은 계열 모델을 여러 번 학습시켜 평균/투표로 예측한다.
   - 대표적으로 Random Forest가 있다.
3. Boosting
   - 이전 모델이 잘 못 맞춘 데이터에 점점 더 집중하면서 순차적으로 학습한다.
   - 대표적으로 AdaBoost, Gradient Boosting, XGBoost 계열이 있다.
4. Stacking
   - 여러 모델의 예측값을 다시 입력값처럼 사용해 최종 메타 모델이 학습한다.

가장 직관적인 Voting부터 시작한다.

# Voting 계열

## VotingClassifier
- hard voting: 각 모델의 최종 예측 클래스를 다수결로 결정한다.
- soft voting: 각 모델의 예측 확률을 평균내어 최종 클래스를 결정한다.

특징
- hard voting은 이해하기 쉽지만, 모델의 확신 정도는 반영하지 못한다.
- soft voting은 확률 정보를 활용하므로 보통 더 유연하다.
- 단, soft voting을 쓰려면 각 모델이 `predict_proba()`를 지원해야 한다.

In [21]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 01. VotingClassifier 

성향이 다른 3개 모델을 묶어 본다.

1. LogisticRegression: 선형 기반, 확률 해석이 비교적 자연스럽다.
2. KNeighborsClassifier: 거리 기반, 주변 샘플의 영향을 많이 받는다.
3. DecisionTreeClassifier: 규칙 분할 기반, 해석이 직관적이다.

서로 다른 방식의 모델을 묶는 이유는 **서로 다른 관점** 을 결합하기 위해서이다.

In [22]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 유방암 데이터 로드
cancer = load_breast_cancer()
X = cancer.data
y = cancer.target

# 훈련/테스트 분리
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 스케일링
# KNN, LogisticRegression은 스케일의 영향을 많이 받는다.
# DecisionTree는 스케일링이 꼭 필요하지는 않지만, 하나의 입력 데이터를 여러 모델이 함께 쓰므로 여기서는 통일해서 전처리한다.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(X_train_scaled.shape, y_train.shape)
print(X_test_scaled.shape, y_test.shape)

(455, 30) (455,)
(114, 30) (114,)


### hard voting

hard voting은 각 모델이 예측한 최종 클래스를 가지고 다수결을 수행한다.

예를 들어,
- lr -> 1
- knn -> 1
- dt -> 0

이라면 최종 예측은 1이다.

즉, **누가 얼마나 확신했는가** 보다 **몇 표를 받았는가** 가 중요하다.

In [23]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import VotingClassifier

# 개별 분류기 준비
lr_clf = LogisticRegression(random_state=42, max_iter=1000)
knn_clf = KNeighborsClassifier(n_neighbors=5)
dt_clf = DecisionTreeClassifier(random_state=42)

# hard voting 분류기 준비
hard_voting_clf = VotingClassifier(
    estimators=[
        ('lr', lr_clf), 
        ('knn', knn_clf),
        ('dt', dt_clf)
    ],
    voting='hard'
)

# 앙상블 학습
hard_voting_clf.fit(X_train_scaled, y_train)

print('hard voting 학습 정확도:', hard_voting_clf.score(X_train_scaled, y_train))
print('hard voting 평가 정확도:', hard_voting_clf.score(X_test_scaled, y_test))

hard voting 학습 정확도: 0.9912087912087912
hard voting 평가 정확도: 0.9824561403508771


### soft voting

soft voting은 각 모델의 예측 확률을 평균내고, 그 평균 확률이 더 큰 클래스를 최종 예측으로 선택한다.

예를 들어 양성 클래스(1)에 대한 확률이
- lr -> 0.90
- knn -> 0.60
- dt -> 0.00

이라면 평균은 `(0.90 + 0.60 + 0.00) / 3 = 0.50` 이다.

즉, soft voting은 다수결뿐 아니라 각 모델의 확신 정도도 함께 반영한다.

In [24]:
from sklearn.metrics import accuracy_score
# 개별 분류기와 hard voting 성능 비교

# 개별 분류기 학습
lr_clf.fit(X_train_scaled, y_train)
knn_clf.fit(X_train_scaled, y_train)
dt_clf.fit(X_train_scaled, y_train)

# 개별 분류기와 하드 보팅 분류기의 예측 결과
lr_pred = lr_clf.predict(X_test_scaled)
knn_pred = knn_clf.predict(X_test_scaled)
dt_pred = dt_clf.predict(X_test_scaled)
hard_voting_pred = hard_voting_clf.predict(X_test_scaled)

# 정확도 비교
compare_df = pd.DataFrame({
    'model' : ['LogisticRegression', 'KNeighborsClassifier', 'DecisionTreeClassifier', 'HardVoting'],
    'accuracy' : [
        accuracy_score(y_test, lr_pred),
        accuracy_score(y_test, knn_pred),
        accuracy_score(y_test, dt_pred),
        accuracy_score(y_test, hard_voting_pred)
    ]
    
})

compare_df

,model,accuracy
0,LogisticRegression,0.982456
1,KNeighborsClassifier,0.956140
2,DecisionTreeClassifier,0.912281
3,HardVoting,0.982456


In [25]:
# 일부 샘플에 대해 각 모델이 어떻게 예측 했는지 확인
start = 0
end = 10

pred_df = pd.DataFrame([
    y_test[start:end],
    lr_pred[start:end],
    knn_pred[start:end],
    dt_pred[start:end],
    hard_voting_pred[start:end]
], index=['true', 'lr', 'knn', 'dt', 'hard voting'])

pred_df

,0,1,2,3,4,5,6,7,8,9
true,0,1,0,1,0,1,1,0,0,0
lr,0,1,0,1,0,1,1,0,0,0
knn,0,1,0,0,0,1,1,0,0,0
dt,0,1,0,1,0,1,1,0,0,0
hard voting,0,1,0,1,0,1,1,0,0,0


In [26]:
# soft voting은 predict_proba 함수를 지원하는 모델로 구성해야 한다.
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import VotingClassifier

# 개별 분류기 준비
lr_clf = LogisticRegression(random_state=42, max_iter=1000)
knn_clf = KNeighborsClassifier(n_neighbors=5)
dt_clf = DecisionTreeClassifier(random_state=42)

# hard voting 분류기 준비
soft_voting_clf = VotingClassifier(
    estimators=[
        ('lr', lr_clf), 
        ('knn', knn_clf),
        ('dt', dt_clf)
    ],
    voting='soft'
)

# 앙상블 학습
soft_voting_clf.fit(X_train_scaled, y_train)

print('soft voting 학습 정확도:', soft_voting_clf.score(X_train_scaled, y_train))
print('soft voting 평가 정확도:', soft_voting_clf.score(X_test_scaled, y_test))

soft voting 학습 정확도: 0.9956043956043956
soft voting 평가 정확도: 0.9649122807017544


In [27]:
from sklearn.metrics import accuracy_score
# 개별 분류기와 soft voting 성능 비교

# 개별 분류기 학습
lr_clf.fit(X_train_scaled, y_train)
knn_clf.fit(X_train_scaled, y_train)
dt_clf.fit(X_train_scaled, y_train)

# 개별 분류기와 하드 보팅 분류기의 예측 결과
lr_pred = lr_clf.predict(X_test_scaled)
knn_pred = knn_clf.predict(X_test_scaled)
dt_pred = dt_clf.predict(X_test_scaled)
soft_voting_pred = soft_voting_clf.predict(X_test_scaled)

# 정확도 비교
compare_df = pd.DataFrame({
    'model' : ['LogisticRegression', 'KNeighborsClassifier', 'DecisionTreeClassifier', 'SoftVoting'],
    'accuracy' : [
        accuracy_score(y_test, lr_pred),
        accuracy_score(y_test, knn_pred),
        accuracy_score(y_test, dt_pred),
        accuracy_score(y_test, soft_voting_pred)
    ]
    
})

compare_df

,model,accuracy
0,LogisticRegression,0.982456
1,KNeighborsClassifier,0.956140
2,DecisionTreeClassifier,0.912281
3,SoftVoting,0.964912


In [28]:
# 일부 샘플에 대해 각 모델이 어떻게 예측 했는지 확인
start = 10
end = 20

# 각 모델의 예측 확률
lr_pred_proba = lr_clf.predict_proba(X_test_scaled)
knn_pred_proba = knn_clf.predict_proba(X_test_scaled)
dt_pred_proba = dt_clf.predict_proba(X_test_scaled)
soft_voting_pred_proba = soft_voting_clf.predict_proba(X_test_scaled)

pred_df = pd.DataFrame([
    y_test[start:end],
    lr_pred_proba[start:end, 1],
    knn_pred_proba[start:end, 1],
    dt_pred_proba[start:end, 1],
    soft_voting_pred_proba[start:end, 1]
], index=['true', 'lr', 'knn', 'dt', 'soft voting'])

pred_df

,0,1,2,3,4,5,6,7,8,9
true,1.000000,0.000000,1.000000,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000
lr,0.989179,0.001332,0.998704,0.000026,0.000276,0.901632,0.365862,0.834998,0.999966,0.983017
knn,1.000000,0.000000,1.000000,0.000000,0.000000,0.800000,0.600000,0.600000,1.000000,0.600000
dt,1.000000,0.000000,1.000000,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000
soft voting,0.996393,0.000444,0.999568,0.000009,0.000092,0.900544,0.655287,0.811666,0.999989,0.861006


## 2. VotingRegressor 회귀 예제

회귀에서는 클래스를 투표하는 대신, 여러 모델의 예측값을 평균내는 방식으로 결합한다.

즉, 분류의 soft voting과 비슷하게 생각할 수 있지만,
최종 출력이 확률이 아니라 연속형 수치라는 점이 다르다.


In [29]:
# 캘리포니아 주택 가격 데이터 로드
df = pd.read_csv('data/california_housing.csv')

# 데이터 구조 확인
print(df.head())
print(df.info())

# 타겟과 피처 분리
X = df.drop('MedHouseVal', axis=1).to_numpy()
y = df['MedHouseVal'].to_numpy()

# 훈련/테스트 분리
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 회귀에서도 KNN은 거리 기반이라 스케일 영향을 받는다.
# 선형회귀 역시 스케일링을 하면 계수 해석 외에는 계산 안정성 측면에서 도움이 될 수 있다.
# 트리는 꼭 필요하지 않지만, 입력 일관성을 위해 함께 사용한다.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(X_train_scaled.shape, y_train.shape)
print(X_test_scaled.shape, y_test.shape)

FileNotFoundError: [Errno 2] No such file or directory: 'data/california_housing.csv'

## 정리

1. 앙상블은 여러 모델의 장점을 결합하려는 전략이다.
2. Hard Voting은 다수결, Soft Voting은 확률 평균이다.
3. 회귀 앙상블은 예측값 평균으로 이해하면 된다.
4. 앙상블 성능은 반드시 개별 모델과 비교해서 판단해야 한다.
5. 서로 다른 성향의 모델을 섞는 것이 앙상블의 핵심 아이디어이다.